# Milestone - Unit 8: the two-tower model -- Unit 5's reader/book factors re-expressed as a learned neural retriever (two PyTorch embedding towers trained by pairwise BPR + weight decay), the SAME reader.book dot product as MF yet the book's new accuracy best (val hit@10 ~0.340 > MF 0.276) because the ranking objective wins on top-k -- provided it is regularized.

## 1. Load the catalog, interactions, and cold readers

The same generated substrate Units 1-7 ran on: the catalog slice, the interaction log (train / val / test splits with implicit 0/1 labels), and the committed cold-reader partition that every scoreboard excludes. No GloVe this time -- the two-tower learns its vectors from the interaction log alone, so the catalog text is not needed to score. `bookrec.generated_dir()` reports a clear error if `recsys/data/generated/` has not been built.

In [ ]:
import json
from collections import defaultdict

import bookrec
import pandas as pd
from bookrec import (
    TwoTowerRetrievalPath,
    catalog_coverage,
    hit_rate_at_k,
    rank,
    run_blended_scoreboard,
    run_validation_scoreboard,
)

K = 10
POOL = 30
SEED = 0

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = cold["cold_readers"]

interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")

{
    "books": len(catalog_ids),
    "interactions": len(rows),
    "cold_readers": len(cold_readers),
}

## 2. Fit the five earlier paths and train the new two-tower (two ~15 s fits)

Six signals, each a `RetrievalPath` with the identical `fit`/`retrieve` contract: the seeded random floor, crowd popularity, the Unit-3 **lexical** content path, the Unit-4 item-item CF, the Unit-5 **matrix factorization** (one ~15 s fit at its pinned config), and the new Unit-8 **two-tower**. The two-tower trains a reader embedding tower and a book embedding tower by **BPR** (the pairwise ranking loss `-log sigmoid(s_pos - s_neg)`) with Adam + **weight decay 1e-4** -- the pinned, regularized config (`wd=0` would overfit below CF). It imports torch lazily inside its own `fit`, seeds it, and runs single-threaded + deterministic, then restores those process globals -- so this is the second ~15 s fit and every seed is fixed, making the whole demo reproducible. We confirm the mean per-epoch BPR loss fell over training.

In [ ]:
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=SEED)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
cf = bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = bookrec.MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)  # ~15 s
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)  # ~15 s, wd=1e-4

registry = bookrec.PathRegistry()
for path in (random_floor, popularity, lexical, cf, mf, two_tower):
    registry.register(path)

# BPR loss falls over training (it does NOT collapse to 0 -- weight decay keeps it honest).
first_loss, last_loss = two_tower.loss_history[0], two_tower.loss_history[-1]
assert last_loss < first_loss

{
    "registered_paths": [path.name for path in registry],
    "reader_tower": list(two_tower.reader_embeddings.shape),
    "book_tower": list(two_tower.item_embeddings.shape),
    "bpr_loss_first_epoch": round(first_loss, 4),
    "bpr_loss_last_epoch": round(last_loss, 4),
}

## 3. The `val` scoreboard -- the two-tower is the book's new best

We score all six paths on the frozen `val` split with `run_validation_scoreboard` (the same 500 eligible readers, the 60 cold readers excluded -- never `test`, which Checkpoint B owns), add catalog coverage from a shared harness, and **assert the harness hit matches the official scoreboard** for every deterministic path, binding these numbers to the shipped code. The two-tower lands at **hit@10 ~0.340 -- the highest single path in the book so far**, honestly above MF's 0.276 (not on par with it).

In [ ]:
known_items = set(catalog_ids)
cold_set = set(cold_readers)

seen_by_reader = defaultdict(set)
relevant_by_reader = defaultdict(set)
for row in rows:
    if row["item_id"] not in known_items or int(row["label"]) != 1:
        continue
    if row["split"] == "train":
        seen_by_reader[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        relevant_by_reader[row["reader_id"]].add(row["item_id"])

eligible_readers = [
    reader_id
    for reader_id in sorted(relevant_by_reader)
    if reader_id not in cold_set and (relevant_by_reader[reader_id] - seen_by_reader[reader_id])
]

single_rows = []
for path in registry:
    board = run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    )
    rec_lists = [
        rank(path.retrieve(rid, {"seen": seen_by_reader[rid]}, K), n=K, exclude=seen_by_reader[rid])
        for rid in eligible_readers
    ]
    coverage = catalog_coverage(rec_lists, catalog_ids)
    harness_hit = sum(
        hit_rate_at_k(recs, relevant_by_reader[rid] - seen_by_reader[rid], K)
        for recs, rid in zip(rec_lists, eligible_readers)
    ) / len(eligible_readers)
    # Deterministic paths reproduce the official scoreboard exactly; the random floor is stateful
    # (its RNG has already advanced inside the scoreboard), so it is exempt from the equality check.
    if path.name != "random":
        assert abs(harness_hit - board.hit_rate_at_k) < 1e-9, path.name
    single_rows.append({
        "path": path.name,
        "hit@10": round(board.hit_rate_at_k, 4),
        "precision@10": round(board.precision_at_k, 4),
        "ndcg@10": round(board.ndcg_at_k, 4),
        "coverage": round(coverage, 4),
    })

scoreboard = {entry["path"]: entry for entry in single_rows}

# The ground-truth numbers the lesson quotes (seed 0, k=10, 500 val readers).
assert abs(scoreboard["random"]["hit@10"] - 0.012) < 0.001
assert abs(scoreboard["popularity"]["hit@10"] - 0.108) < 0.001
assert abs(scoreboard["lexical"]["hit@10"] - 0.158) < 0.001
assert abs(scoreboard["item-item"]["hit@10"] - 0.252) < 0.001
assert abs(scoreboard["mf"]["hit@10"] - 0.276) < 0.001
assert abs(scoreboard["two-tower"]["hit@10"] - 0.340) < 0.001
# The two-tower is strictly the best single path -- honestly above MF, not on par with it.
assert scoreboard["two-tower"]["hit@10"] == max(e["hit@10"] for e in single_rows)
assert scoreboard["two-tower"]["hit@10"] > scoreboard["mf"]["hit@10"]

pd.DataFrame(single_rows)

Read it honestly. The two-tower reaches **hit@10 ~0.340 -- the book's new best**, clear of MF (0.276), CF (0.252), lexical (0.158), popularity (0.108) and the random floor (0.012). The architecture is **exactly MF's model** -- a reader embedding dotted with a book embedding -- so the win is not a new *model*. What changed is the **objective and optimizer**: a pairwise **BPR ranking** loss trained with Adam mini-batches + weight decay, against MF's pointwise logistic full-batch gradient descent. A ranking objective optimizes directly for ordering the top-k, and here it wins -- **but only because it is regularized**: at `weight_decay=0` the BPR loss collapses toward 0 (it memorizes the train pairs) and val hit@10 falls below CF. Do not read 'two-tower > MF' as 'neural is always better'; read it as 'the ranking objective + weight decay wins here'. Notice the cost, too: the two-tower's catalog coverage (~0.177) is narrower than CF's (~0.49) -- it concentrates on a confident core.

## 4. One reader's two-tower recs, and the score IS a dot product of learned embeddings

For a single reader we read the top two-tower recommendations, then rebuild each score by hand to show the retriever is nothing but `reader_embedding . book_embedding`. Because we fit with `catalog=catalog_ids`, the book tower's rows line up with the sorted catalog; the reader tower's rows line up with the sorted readers who have at least one train positive. So we can look up the reader's learned vector, dot it against every book vector, and confirm it reproduces the retrieve scores exactly -- the same dot-product retriever as Unit 5's MF and Unit 7's GloVe cosine, now learned neurally.

In [ ]:
reader_id = 3
seen = seen_by_reader[reader_id]
reader_recs = rank(two_tower.retrieve(reader_id, {"seen": seen}, 5), n=5, exclude=seen)

# Public row alignment: book-tower rows follow the sorted catalog (we fit with catalog=catalog_ids);
# reader-tower rows follow the sorted readers that have >= 1 train positive.
train_readers = sorted({row["reader_id"] for row in rows
                        if row["split"] == "train" and int(row["label"]) == 1
                        and row["item_id"] in known_items})
reader_row = {rid: idx for idx, rid in enumerate(train_readers)}
book_col = {item_id: idx for idx, item_id in enumerate(catalog_ids)}

reader_vec = two_tower.reader_embeddings[reader_row[reader_id]]
book_matrix = two_tower.item_embeddings
hand_scores = reader_vec @ book_matrix.T  # one dot product per catalog book

# Every retrieve score is exactly the reader.book embedding dot product.
for candidate in reader_recs:
    assert abs(candidate.score - float(hand_scores[book_col[candidate.item_id]])) < 1e-5

{
    "reader": reader_id,
    "reader_vector_dim": len(reader_vec),
    "recommended": [
        {
            "book": candidate.item_id,
            "retrieve_score": round(candidate.score, 4),
            "hand_dot_product": round(float(hand_scores[book_col[candidate.item_id]]), 4),
        }
        for candidate in reader_recs
    ],
}

## 5. Add the two-tower to the Unit-6 pinned blend -- hit up, coverage down

Finally the ablation. We take the **Unit-6 pinned blend** -- weights `{popularity 0.25, lexical 0.5, item-item 1.0, mf 1.0}` at per-path pool 30, the blend Unit 6 shipped and Checkpoint B froze -- then add the two-tower (weight 0.5) and re-measure hit@10 and catalog coverage with `run_blended_scoreboard`. Unlike Unit 7's weak-but-diverse semantic path (which lifted both hit *and* coverage), the two-tower is the opposite kind of signal: strong but **concentrated** (coverage ~0.177). So adding it **sharpens the blend's hit (0.306 -> 0.314) but trims its coverage (0.333 -> 0.310)** -- an honest trade, not a free win.

In [ ]:
PINNED_WEIGHTS = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}
blend_paths = {"popularity": popularity, "lexical": lexical, "item-item": cf, "mf": mf}

board_without = run_blended_scoreboard(
    blend_paths, interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL,
    weights=PINNED_WEIGHTS, cold_readers=cold_readers,
)
board_with = run_blended_scoreboard(
    {**blend_paths, "two-tower": two_tower}, interactions_path, catalog_ids=catalog_ids, k=K,
    pool=POOL, weights={**PINNED_WEIGHTS, "two-tower": 0.5}, cold_readers=cold_readers,
)

# Measured (Unit-6 pinned blend, pool 30): hit 0.306 -> 0.314, coverage 0.333 -> 0.310.
assert abs(board_without.hit_rate_at_k - 0.306) < 0.003
assert abs(board_without.catalog_coverage - 0.333) < 0.003
assert abs(board_with.hit_rate_at_k - 0.314) < 0.003
assert abs(board_with.catalog_coverage - 0.310) < 0.003
# The strong-but-concentrated two-tower lifts hit but costs coverage -- the honest trade.
assert board_with.hit_rate_at_k > board_without.hit_rate_at_k
assert board_with.catalog_coverage < board_without.catalog_coverage

print(f"blend WITHOUT two-tower: hit@10={board_without.hit_rate_at_k:.4f} "
      f"coverage={board_without.catalog_coverage:.4f}")
print(f"blend WITH    two-tower: hit@10={board_with.hit_rate_at_k:.4f} "
      f"coverage={board_with.catalog_coverage:.4f}")

**The honest headline.** The two-tower is the strongest single path in the book (val hit@10 ~0.340) -- the *same* reader.book dot product as Unit 5's MF, re-expressed as a learned neural retriever and trained with a pairwise BPR ranking objective + weight decay. Dropped into the Unit-6 blend it raises hit@10 (0.306 -> 0.314) but narrows coverage (0.333 -> 0.310), because a confident, concentrated signal trades breadth for precision -- the mirror image of Unit 7's weak-but-diverse semantic path.

**The bridge.** MF (numpy gradient descent), GloVe (pretrained text), and the two-tower (learned neural) are all **dot-product retrievers over embeddings** -- one retrieval idea, three kinds of vector. What a neural tower buys that numpy MF does not is a place to attach **features**: Unit 9 replaces the bare ID lookup with *feature* towers so a brand-new book -- with no interaction history yet -- can still be scored from its content, and Units 11-12 build the reranker and sequence model on this learned stack.